# M7 · Lab 2 — MLflow Model Registry
### Module 7 | Spine: Truck Delay Classification

| | |
|---|---|
| **Duration** | 60 min · **Difficulty** Intermediate · **Tier 3** |
| **Tools** | Python 3.12.10, `mlflow`, the real M3 model |
| **Prerequisite** | The MLflow Tracking Server (instructor_setup) — `MLFLOW_TRACKING_URI` set |
| **Builds toward** | M8 pipeline registers + promotes models automatically |

## 💻 Where to run this
Run on the **same SageMaker instance from M6** (auto-stops overnight — just **Start** it). New deps this module install in
the setup cell. Portable fallback: Colab / local Python 3.12.10. The **real** M3 artifacts ship in `data/` — nothing synthetic.

## Why a registry (not `joblib.dump` to S3)?
A `.pkl` on S3 has no **version history**, no **stage** ("is this the one in prod?"), no **approval gate**, no **audit
trail**, and no clean **rollback**. The MLflow **Model Registry** adds all of it: every model is a versioned entry that
moves through **Staging → Production → Archived**, with who-promoted-what recorded. That governance is the difference
between "a model file" and "a production asset."

## Step 1 · Setup + connect to the tracking server

In [ ]:
# import sys, subprocess
# def _pip(*p): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *p], check=True)
# try:
#     import mlflow
# except ImportError:
#     _pip("mlflow==2.14.1", "xgboost==2.0.3"); import mlflow
# from mlflow.tracking import MlflowClient

import os
import mlflow
TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://localhost:5000")
mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment("truck-delay-classification")
print("Tracking:", TRACKING_URI)

In [ ]:
import os, json
import numpy as np, pandas as pd

DATA_DIR  = os.environ.get("M7_DATA_DIR", "data")
REF_CSV   = os.path.join(DATA_DIR, "reference", "final_features.csv")
ART_DIR   = os.path.join(DATA_DIR, "artifacts")
assert os.path.exists(REF_CSV), f"Real reference frame missing at {REF_CSV} (ships in Module 7/labs/data/)."

ref = pd.read_csv(REF_CSV)
fmeta = json.load(open(os.path.join(DATA_DIR, "reference", "feature_metadata.json")))
print("Reference:", ref.shape, "| target rate:", round(ref[fmeta["target"]].mean(), 4))

In [ ]:
# Reuse the EXACT M3 inference preprocessing (scale -> one-hot -> passthrough -> reorder to 128 features)
import joblib
_model   = joblib.load(os.path.join(ART_DIR, "xgboost_model.pkl"))
_encoder = joblib.load(os.path.join(ART_DIR, "encoder.pkl"))
_scaler  = joblib.load(os.path.join(ART_DIR, "scaler.pkl"))
mmeta = json.load(open(os.path.join(ART_DIR, "model_metadata.json")))
M_CONT, M_CAT, M_BIN, M_FEATS = (mmeta["continuous_cols"], mmeta["categorical_cols"],
                                 mmeta["binary_ordinal_cols"], mmeta["feature_names"])

def to_model_matrix(df):
    x_cont = pd.DataFrame(_scaler.transform(df[M_CONT]), columns=M_CONT)
    x_cat  = pd.DataFrame(_encoder.transform(df[M_CAT]), columns=_encoder.get_feature_names_out(M_CAT))
    x_bin  = df[M_BIN].reset_index(drop=True)
    return pd.concat([x_cont, x_cat, x_bin], axis=1)[M_FEATS]
print("Preprocessing helper ready — produces the 128-feature model matrix.")

## Step 2 · Log the real model as a run, registering it as `truck-delay-classifier`
We log the genuine M3 model + its true test metrics (from `model_metadata.json`) + the encoder/scaler as artifacts, and
**register** it in one call. No retraining — this is the exact model M5 deployed and M6 monitored.

In [ ]:
with mlflow.start_run(run_name="m3-baseline-xgboost") as run:
    mlflow.log_params({"model": "XGBoost", "n_features": mmeta["n_features"],
                       "training_rows": mmeta["training_rows"]})
    mlflow.log_metrics({f"test_{k.replace('test_', '')}": v
                        for k, v in mmeta["test_metrics"].items()})
    mlflow.log_dict(mmeta, "model_metadata.json")
    mlflow.xgboost.log_model(_model, artifact_path="model",
                             registered_model_name="truck-delay-classifier")
print("Logged + registered. Run:", run.info.run_id)

## Step 3 · Inspect the registered versions

In [ ]:
client = MlflowClient()
for mv in client.search_model_versions("name='truck-delay-classifier'"):
    print(f"  version {mv.version}  stage={mv.current_stage}  run={mv.run_id[:8]}")
latest = max(int(mv.version) for mv in client.search_model_versions("name='truck-delay-classifier'"))
print("Latest version:", latest)

## Step 4 · Stage transitions — Staging → Production → Archived
This is the governance gate. In a real org, promotion to **Production** is an approval step (a human, or the M8 pipeline's
condition step) — not a `cp` command.

In [ ]:
client.transition_model_version_stage("truck-delay-classifier", latest, stage="Staging")
print("-> Staging (validation candidate)")
client.transition_model_version_stage("truck-delay-classifier", latest, stage="Production",
                                      archive_existing_versions=True)
print("-> Production (archives any previous Production version automatically)")

## Step 5 · Load **from the registry by stage** and run real inference

In [ ]:
prod_model = mlflow.pyfunc.load_model("models:/truck-delay-classifier/Production")
sample = ref.sample(500, random_state=7)
X = to_model_matrix(sample)
preds = prod_model.predict(X)
from sklearn.metrics import f1_score
print("Loaded Production model from registry. f1 on 500 real rows:",
      round(f1_score(sample[fmeta["target"]], preds), 4))

## Step 6 · Rollback drill
Because every version is retained, rollback is one call — archive the bad version, re-promote the previous one. Try it,
then put Production back.

In [ ]:
# (Demonstration) archive current Production, and you would re-promote the prior version here.
print("Current stages:")
for mv in client.search_model_versions("name='truck-delay-classifier'"):
    print(f"  v{mv.version}: {mv.current_stage}")
print("\nRollback = transition the previous version back to Production. Audit trail is preserved in the run history.")

## Verification Checklist
- [ ] MLflow UI (`$MLFLOW_TRACKING_URI`) shows the `truck-delay-classification` experiment + the registered model
- [ ] `truck-delay-classifier` has at least one version in **Production**
- [ ] You loaded `models:/truck-delay-classifier/Production` and scored real rows (f1 ≈ 0.67 on a fresh sample)
- [ ] You can explain Staging vs Production vs Archived and why promotion is an approval gate

## What's next — Lab 3
MLflow tracks *your* runs on *your* server. **W&B** is the hosted alternative with first-class **hyperparameter sweeps** —
Lab 3 runs a sweep and compares the two tools head-to-head.

## Troubleshooting
| Symptom | Fix |
|---|---|
| `Connection refused` to tracking URI | the MLflow EC2 isn't up / SG blocks :5000 — see instructor_setup; allow ~2 min after deploy |
| `mlflow.xgboost` missing | `pip install xgboost`; the model is an XGBoost sklearn-API classifier |
| stage transition deprecation warning | MLflow 2.x is moving to *aliases*; **stages** still work in 2.14 and are what the course teaches |